# FSP-CED: computational study (Colab) — code version 3, revision 1

Fire Scheduling Problem with Common and Expandable Deadlines — Canut-de-Bon, Catalán & Yuraszeck.

**Before running (once):** in Google Drive, replace `MyDrive/FSP_CED/fspced_code.zip` with the new zip and upload this
notebook next to it. **Do not delete `MyDrive/FSP_CED/results/`**: every run already saved is kept and skipped; only
the new experiments are run.
The Gurobi license is taken from Colab Secrets (`WLSACCESSID`, `WLSSECRET`, `LICENSEID`) or from a `gurobi.lic` file in
`MyDrive/FSP_CED/` or `MyDrive/mbh_toptw/`. Without a license, HiGHS is used (slower).

**What is new in this revision (answers to the review) — only cells 1–4b, 7f and 8 need to be run**

| | |
|---|---|
| `IG+CP`, `IG+CPF` | CP-P and CP-F started from the plan of IG-ED, the same start as LBBD (fair comparison; cell 7f) |
| `root_bounds.py` | linear and integer bounds of the master problem with and without the inequalities (cell 7f) |
| `sensitivity.py` | gain of the expandable deadline under other caps and extensions (coastal set; cell 7f) |
| `instances_dispersed/` | fifth instance set: 24 instances with arbitrary battery sets, capped and short windows (cell 7f) |
| `analyze.py` | new tables `tab_root`, `tab_warm`, `tab_disp`, paired test by instance, sensitivity macros (cell 8) |

Everything else is unchanged: `LBBD`, `CPF`, `CPS`, `CPF0`, the ablation variants `LBBD-T/-P/-B/-N/-BN/-0/-S`, and the
first four instance sets. Cells 5–7e find their runs already saved and skip them.

**How it runs:** the code zip is unpacked on Colab's local disk at every session, while results go to
`MyDrive/FSP_CED/results/`. Every finished run is saved at once; **if the session stops, run the cells again and it
continues where it left off**.

**Smoke test first (cell 4b):** about 15 minutes; it must print `SMOKE TEST OK`. Its records go to a scratch folder.

`SESSION_BUDGET_H` (cell 3) stops launching new runs when the session has lasted that long, so that everything is saved
before Colab ends the session (about 12 h in the free version).

**Splitting the work:** make copies of this notebook (*File → Save a copy in Drive*) and set `SHARD = '0/4'`, `'1/4'`,
`'2/4'`, `'3/4'` in cell 3 of each; each copy writes its own files. Run cell 8 once, in any session, when all are done.

**Instance sets:** `instances/` (coastal benchmark, 54), `instances_chakim/` (design of Cha & Kim 2010, 54),
`instances_large/` (200–300 targets, 18), `instances_short/` (short windows, 36, ED only), `instances_dispersed/`
(arbitrary battery sets, 24, ED only).
**New compute (cell 7f)** with the default limits: at most about 30 h of a 2-vCPU VM if every run reached its limit,
probably 12–18 h. With `SHARD` in four sessions it fits in one day.


In [ ]:
# 1. Google Drive, code (local copy) and results (in Drive)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, time
WORK = '/content/drive/MyDrive/FSP_CED'            # holds fspced_code.zip, this notebook and results/
CODE = '/content/fsp'                              # local copy of the code, unpacked at every session
assert os.path.exists(f'{WORK}/fspced_code.zip'), 'Upload fspced_code.zip to MyDrive/FSP_CED/'
shutil.rmtree(CODE, ignore_errors=True)
shutil.unpack_archive(f'{WORK}/fspced_code.zip', CODE)
assert os.path.exists(f'{CODE}/scripts/root_bounds.py'), 'This is not the zip of the revision: replace fspced_code.zip in Drive'
os.makedirs(f'{WORK}/results', exist_ok=True)
os.symlink(f'{WORK}/results', f'{CODE}/results')   # scripts write to results/ -> Drive
%cd {CODE}
print('code:', sorted(os.listdir(CODE)))
print('runs already saved:', {f: sum(1 for _ in open(f'{WORK}/results/{f}')) for f in sorted(os.listdir(f'{WORK}/results')) if f.endswith('.jsonl')})

In [ ]:
# 2. Solvers (pinned versions used in the paper) and Gurobi WLS license
!pip -q install ortools==9.15.6755 highspy numba "gurobipy>=12"
import shutil
# license: Colab Secrets (WLSACCESSID, WLSSECRET, LICENSEID) or a gurobi.lic file in Drive
# (looked up in this project's folder and in the folder of the MBH-TOPTW paper)
LIC_PATH = '/content/gurobi.lic'
try:
    from google.colab import userdata
    acc, sec, lid = userdata.get('WLSACCESSID'), userdata.get('WLSSECRET'), userdata.get('LICENSEID')
    with open(LIC_PATH, 'w') as f:
        f.write(f"WLSACCESSID={acc}\nWLSSECRET={sec}\nLICENSEID={lid}\n")
    print('license written from Colab Secrets')
except Exception as e:
    for cand in [f'{WORK}/gurobi.lic', '/content/drive/MyDrive/mbh_toptw/gurobi.lic']:
        if os.path.exists(cand):
            shutil.copy(cand, LIC_PATH); print('license copied from', cand); break
    else:
        print('NO GUROBI LICENSE FOUND: the MILPs and the LBBD master will use HiGHS (slower)')
os.environ['GRB_LICENSE_FILE'] = LIC_PATH
# full-license test (the pip license is limited to 2000 variables)
!python -c "import gurobipy as gp; m=gp.Model(); m.setParam('OutputFlag',0); x=m.addVars(5000, ub=1.0); m.setObjective(x.sum(), -1); m.optimize(); print('Gurobi', gp.gurobi.version(), 'full license OK:', m.Status==2)"
!python -c "from fspced.milp import default_backend; print('MILP backend:', default_backend())"
!nproc; free -g | head -2

In [ ]:
# 3. Configuration
TL_CP    = 300     # s, CP-P, CP-F, IG+CP and LBBD (whole run)
TL_MILP  = 300     # s, time-indexed MILP (TI)
TL_DJ    = 60      # s, disjunctive MILP (DJ, baseline)
TL_HEUR  = 10      # s, GA and IG-ED per run
TL_LARGE = 300     # s, CP-P, CP-F and LBBD on the large set (200-300 targets)
SEEDS    = '0,1,2' # independent runs of GA and IG-ED
IG_TIME  = 5       # s, IG-ED phase inside IG+CP and LBBD
WORKERS  = 2       # CP-SAT workers (= vCPUs)
SHARD    = '0/1'   # e.g. '0/4', '1/4', '2/4', '3/4' in four sessions
SESSION_BUDGET_H = 10.5   # hours: no new runs are launched after this (Colab free ends sessions at ~12 h)
ABL = 'LBBD-T,LBBD-P,LBBD-B,LBBD-N,LBBD-BN,LBBD-0,LBBD-S'   # ablation variants of the decomposition
tag = SHARD.replace('/', 'of')
os.environ['FSP_T0'] = str(time.time()); os.environ['FSP_BUDGET_H'] = str(SESSION_BUDGET_H)

In [ ]:
# 4. Instances (shipped with the code; regenerated deterministically if missing) and quick checks
if not os.path.exists('instances/m3_n10_t30_r1.json'):
    !python scripts/gen_instances.py instances
if not os.path.exists('instances_short/sw_m3_n20_t05_r1.json'):
    !python scripts/gen_instances.py instances_short --short
if not os.path.exists('instances_dispersed/dp_m6_n40_c30_r1.json'):
    !python scripts/gen_instances.py instances_dispersed --dispersed
!ls instances instances_chakim instances_large instances_short instances_dispersed | grep -c json
!python scripts/verify_single.py 300
!python scripts/verify_theory.py --n 40

In [ ]:
# 4b. SMOKE TEST (about 15 min): every script on a few small instances and on the smallest large one, short limits.
#     Writes to /content/fsp/smoke (NOT to the results in Drive). Run it once per new code version, and go on with
#     cells 5-8 only if it ends with "SMOKE TEST OK". If it fails, send me the file smoke/smoke.log.
!python scripts/smoke_test.py --out smoke --workers {WORKERS}

In [ ]:
# 5. Heuristics: MWSPT, GA, IG-ED   (already done in the first campaign: skipped)
!python scripts/run_heur.py --inst instances --out results/heur_{tag}.jsonl --tl {TL_HEUR} --seeds {SEEDS} --shard {SHARD}

In [ ]:
# 6. MILPs: time-indexed (TI) on all instances, disjunctive (DJ) up to n=20, and LP relaxations   (already done: skipped)
!python scripts/run_milp.py --inst instances --out results/milp_{tag}.jsonl --tl {TL_MILP} --tl-dj {TL_DJ} --threads {WORKERS} --nmax-ti 1000 --nmax-dj 20 --shard {SHARD}

In [ ]:
# 7. Coastal benchmark: CP-P and IG+CP (already done: skipped), CP-F and the new LBBD;
#    then the two CP models with and without the redundant constraints (ED only; CPS = CP-P with them, CPF0 = CP-F without)
!python scripts/run_cp.py --inst instances --out results/cp_{tag}.jsonl --tl {TL_CP} --workers {WORKERS} --ig-time {IG_TIME} --methods CP,CPF,IG+CP,LBBD --shard {SHARD}
!python scripts/run_cp.py --inst instances --out results/cp_{tag}.jsonl --tl {TL_CP} --workers {WORKERS} --ig-time {IG_TIME} --methods CPS,CPF0 --variants ED --shard {SHARD}

In [ ]:
# 7b. Large set (18 instances with 200-300 targets): CP-P (done), CP-F and LBBD; MWSPT and IG-ED (done)
if not os.path.exists('instances_large/m9_n200_t30_r1.json'):
    !python scripts/gen_instances.py instances_large --large
!python scripts/run_cp.py --inst instances_large --out results/large_cp_{tag}.jsonl --tl {TL_LARGE} --workers {WORKERS} --ig-time {IG_TIME} --methods CP,CPF,LBBD --shard {SHARD}
!python scripts/run_heur.py --inst instances_large --out results/large_heur_{tag}.jsonl --tl {TL_HEUR} --seeds 0 --shard {SHARD}

In [ ]:
# 7c. Instances following the published design of Cha and Kim (2010): heuristics and TI (done), CP-P (done), CP-F and LBBD
if not os.path.exists('instances_chakim/ck_m3_n18_p20_r1.json'):
    !python scripts/gen_instances.py instances_chakim --chakim
!python scripts/run_heur.py --inst instances_chakim --out results/ck_heur_{tag}.jsonl --tl {TL_HEUR} --seeds {SEEDS} --shard {SHARD}
!python scripts/run_milp.py --inst instances_chakim --out results/ck_milp_{tag}.jsonl --tl {TL_MILP} --tl-dj {TL_DJ} --threads {WORKERS} --nmax-ti 1000 --nmax-dj 0 --shard {SHARD}
!python scripts/run_cp.py --inst instances_chakim --out results/ck_cp_{tag}.jsonl --tl {TL_CP} --workers {WORKERS} --ig-time {IG_TIME} --methods CP,CPF,LBBD --shard {SHARD}

In [ ]:
# 7d. NEW: short-window set (36 instances, expandable deadline only): heuristics, TI, CP-P, CP-F and LBBD
!python scripts/run_heur.py --inst instances_short --out results/short_heur_{tag}.jsonl --tl {TL_HEUR} --seeds {SEEDS} --variants ED --shard {SHARD}
!python scripts/run_cp.py --inst instances_short --out results/short_cp_{tag}.jsonl --tl {TL_CP} --workers {WORKERS} --ig-time {IG_TIME} --methods LBBD,CPF,CP --variants ED --shard {SHARD}
!python scripts/run_milp.py --inst instances_short --out results/short_milp_{tag}.jsonl --tl {TL_MILP} --tl-dj {TL_DJ} --threads {WORKERS} --nmax-ti 1000 --nmax-dj 0 --variants ED --shard {SHARD}

In [ ]:
# 7e. NEW: structural results on random instances, and ablation of the decomposition (ED variant only)
#     LBBD-T: the subproblem does not receive the modes of the master; LBBD-P: pairwise subproblem;
#     LBBD-B: no Johnson/ladder inequalities; LBBD-N: no-good cuts (theta = 1); LBBD-BN: both;
#     LBBD-0: conventional design (all of the above); LBBD-S: classical cuts (NOT valid, measured on purpose).
#     Kept in separate files because LBBD-S can certify wrong bounds.
if SHARD.startswith('0/'):
    !python scripts/verify_theory.py --n 300 --out results/theory.jsonl
!python scripts/run_cp.py --inst instances_short --out results/short_ablation_{tag}.jsonl --tl {TL_CP} --workers {WORKERS} --ig-time {IG_TIME} --methods {ABL} --variants ED --shard {SHARD}
!python scripts/run_cp.py --inst instances --out results/ablation_{tag}.jsonl --tl {TL_CP} --workers {WORKERS} --ig-time {IG_TIME} --methods {ABL} --variants ED --shard {SHARD}
!python scripts/run_cp.py --inst instances_large --out results/large_ablation_{tag}.jsonl --tl {TL_LARGE} --workers {WORKERS} --ig-time {IG_TIME} --methods {ABL} --variants ED --shard {SHARD}
!python scripts/run_cp.py --inst instances_chakim --out results/ck_ablation_{tag}.jsonl --tl {TL_CP} --workers {WORKERS} --ig-time {IG_TIME} --methods {ABL} --variants ED --shard {SHARD}

In [ ]:
# 7f. REVISION (new): experiments added in answer to the review. Everything is resumable.
#  (1) CP-P and CP-F started from the plan of IG-ED (IG+CP, IG+CPF), ED and CD, on the first four sets
#      (IG+CP on the coastal set is already saved and skipped)
!python scripts/run_cp.py --inst instances --out results/cp_{tag}.jsonl --tl {TL_CP} --workers {WORKERS} --ig-time {IG_TIME} --methods IG+CP,IG+CPF --shard {SHARD}
!python scripts/run_cp.py --inst instances_chakim --out results/ck_cp_{tag}.jsonl --tl {TL_CP} --workers {WORKERS} --ig-time {IG_TIME} --methods IG+CP,IG+CPF --shard {SHARD}
!python scripts/run_cp.py --inst instances_short --out results/short_cp_{tag}.jsonl --tl {TL_CP} --workers {WORKERS} --ig-time {IG_TIME} --methods IG+CP,IG+CPF --variants ED --shard {SHARD}
!python scripts/run_cp.py --inst instances_large --out results/large_cp_{tag}.jsonl --tl {TL_LARGE} --workers {WORKERS} --ig-time {IG_TIME} --methods IG+CP,IG+CPF --shard {SHARD}
#  (2) dispersed set (24 instances, arbitrary battery sets, ED only): every method, and the ablation
!python scripts/run_heur.py --inst instances_dispersed --out results/disp_heur_{tag}.jsonl --tl {TL_HEUR} --seeds {SEEDS} --variants ED --shard {SHARD}
!python scripts/run_cp.py --inst instances_dispersed --out results/disp_cp_{tag}.jsonl --tl {TL_CP} --workers {WORKERS} --ig-time {IG_TIME} --methods LBBD,CPF,CP,IG+CP,IG+CPF --variants ED --shard {SHARD}
!python scripts/run_milp.py --inst instances_dispersed --out results/disp_milp_{tag}.jsonl --tl {TL_MILP} --tl-dj {TL_DJ} --threads {WORKERS} --nmax-ti 1000 --nmax-dj 0 --variants ED --shard {SHARD}
!python scripts/run_cp.py --inst instances_dispersed --out results/disp_ablation_{tag}.jsonl --tl {TL_CP} --workers {WORKERS} --ig-time {IG_TIME} --methods {ABL} --variants ED --shard {SHARD}
#  (3) bounds of the master problem with and without the inequalities (HiGHS, deterministic; about 1 h in total)
for d, pre in [('instances', ''), ('instances_chakim', 'ck_'), ('instances_short', 'short_'), ('instances_large', 'large_'), ('instances_dispersed', 'disp_')]:
    !python scripts/root_bounds.py --inst {d} --out results/{pre}root_{tag}.jsonl --tl 60 --shard {SHARD}
#  (4) gain of the expandable deadline under other caps and extensions (coastal set, LBBD, 120 s per run)
!python scripts/sensitivity.py --inst instances --out results/sensitivity_{tag}.jsonl --tl 120 --workers {WORKERS} --shard {SHARD}


In [ ]:
# 8. Tables, macros and figures for the paper (run once, after all shards are done)
!python scripts/analyze.py --res results --out paper_results
!python scripts/example.py paper_results/Figure_2.pdf
shutil.copytree('paper_results', f'{WORK}/paper_results', dirs_exist_ok=True)
shutil.make_archive(f'{WORK}/paper_results', 'zip', 'paper_results')
shutil.make_archive(f'{WORK}/results_fsp', 'zip', f'{WORK}/results')
from google.colab import files
files.download(f'{WORK}/paper_results.zip'); files.download(f'{WORK}/results_fsp.zip')